# FX Cross-Country Signal Assessment

This notebook walks through the complete pipeline for evaluating **pre-aggregated FX sentiment indices** as predictors of FX price returns across multiple countries.

> **Disclaimer:** This notebook is provided for informational and research purposes only. Nothing here constitutes financial advice or a recommendation to buy, sell, or hold any asset. Sentiment indicators derived here reflect aggregated model outputs and should not be used as the sole basis for any investment or policy decision.

## What This Notebook Does

1. **Loads** hourly FX sentiment index data from zip archives (one per country)
2. **Aggregates** hourly indices into daily sentiment per topic, deriving **3 sentiment metrics** from the raw columns
3. **Loads** FX price data and computes forward returns at multiple horizons (7d, 1m, 3m)
4. **Correlates** all 3 metrics × rolling transforms (z-score, mean) × 3 lookback windows with forward FX returns across 3 regimes (BOTH, UP, DOWN)
5. **Applies** multiple-testing corrections (Benjamini-Hochberg FDR) to identify statistically significant drivers
6. **Validates** signals with a 60/40 walk-forward in-sample / out-of-sample split
7. **Ranks** countries, sentiment groups, and metrics in a cross-country comparison table

## Sentiment Metrics

The raw hourly data contains 3 columns (`publication_time`, `headline_count`, `sentiment_sum`). From these we derive **3 daily metrics**:

| Metric | Formula | What it measures |
|---|---|---|
| `sentiment_avg` | `sentiment_sum / headline_count` | Average sentiment direction (bullish/bearish) |
| `sentiment_std_daily` | std of hourly `sentiment_avg` values | Intra-day disagreement / dispersion |
| `headline_count` | sum of hourly counts | Attention / news volume |

## Analysis Parameters

| Parameter | Value | Rationale |
|---|---|---|
| Return horizons | 7d, 1m, 3m | Short-to-medium-term FX signal windows |
| Rolling windows | 10, 21, 63 days | Weekly, monthly, quarterly smoothing |
| Transforms | rolling z-score, rolling mean | Z-score is stationary; mean captures trends |
| Regimes | BOTH, UP, DOWN | Based on trailing 63-day FX return sign |
| Walk-forward | 60% IS / 40% OOS | Single temporal split for persistence validation |
| Significance | BH FDR at q=0.05, per regime | Controls false discovery rate within each regime |

## Input Data

**FX Sentiment Indices** (`data/regional_macro/*.zip`): Each zip contains gzipped CSVs partitioned by `topic_name` and `index_type` (DOMESTIC / INTERNATIONAL). Each row is one hourly observation:

| Column | Description |
|---|---|
| `publication_time` | Hourly timestamp |
| `headline_count` | Number of headlines in this hour |
| `sentiment_sum` | Sum of sentiment scores across headlines |

**FX Prices** (`data/fx_prices.csv`): Daily prices with columns `pair`, `date`, `price`. Each country is mapped to its primary FX pair via `COUNTRY_TO_FX_PAIR`.

## Zip File Format

Each country has a zip archive in `FX_SENTIMENT_DIR`, named using the pattern:

```
macro_geopolitical_sentiment_{uuid}_{Country Name}.zip
├── topic_name=Economic Data-Inflation/
│   ├── index_type=DOMESTIC/
│   │   └── part-XXXXX-{uuid}.c000.csv.gz
│   └── index_type=INTERNATIONAL/
│       └── part-XXXXX-{uuid}.c000.csv.gz
└── ...  (26 topic partitions × 2 index types = ~104 part-files per country)
```

**Index types:** `DOMESTIC` (headlines published within the country) and `INTERNATIONAL` (headlines from outside). No `COMBINED` partition exists in this format.

## Step 0: Configuration & Imports

Set up paths, countries to analyse, FX pair mappings, and the key parameters that control the correlation analysis.

In [ ]:
%pip install -q -U pandas numpy scipy


In [ ]:
import warnings, zipfile, gzip, time
from pathlib import Path
from IPython.display import display as ipy_display

import numpy as np
import pandas as pd
from scipy.stats import t as t_dist

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

# ── Paths ──────────────────────────────────────────────────────────────────────
FX_SENTIMENT_DIR = Path("data/regional_macro")  # zip archives, one per country
FX_PRICES_FILE = Path("data/fx_prices.csv")  # long format: pair, date, price

# ── Countries to analyse ───────────────────────────────────────────────────────
COUNTRIES = [
    "united states",
    "china",
    "germany",
    "united kingdom",
    "japan",
    "france",
    "australia",
]

COUNTRY_LABELS = {
    "united states": "US",
    "china": "China",
    "germany": "Germany",
    "united kingdom": "UK",
    "japan": "Japan",
    "france": "France",
    "australia": "Australia",
}

# Maps each country to the FX ticker in fx_prices.csv.
# Tickers are currency indices: AUD_IND, CNY_IND, EUR_IND, GBP_IND, JPY_IND, USD_IND.
COUNTRY_TO_FX_PAIR = {
    "united states": "USD_IND",
    "china": "CNY_IND",
    "germany": "EUR_IND",
    "united kingdom": "GBP_IND",
    "japan": "JPY_IND",
    "france": "EUR_IND",
    "australia": "AUD_IND",
}

# ── Analysis parameters ────────────────────────────────────────────────────────
RETURN_HORIZONS = {"7d": 7, "1m": 21, "3m": 63}
ROLLING_WINDOWS = [10, 21, 63]
SIGNIFICANCE_LEVEL = 0.05
REGIME_LOOKBACK = 63
REGIMES = ["BOTH", "UP", "DOWN"]
SPARSE_THRESHOLD_PCT = 15
WALK_FORWARD_IS_FRACTION = 0.60

# Index types present in the zip archives.
INDEX_TYPES = ["DOMESTIC", "INTERNATIONAL"]

SENTIMENT_METRICS = [
    "sentiment_avg",
    "sentiment_std_daily",
    "headline_count",
]

print(f"Countries       : {COUNTRIES}")
print(f"FX pairs        : {sorted(set(COUNTRY_TO_FX_PAIR.values()))}")
print(f"Sentiment metrics: {SENTIMENT_METRICS}")
print(f"Return horizons : {list(RETURN_HORIZONS.keys())}")
print(f"Rolling windows : {ROLLING_WINDOWS}")
print(f"Regimes         : {REGIMES}  (lookback={REGIME_LOOKBACK}d)")
print(f"Walk-forward    : {WALK_FORWARD_IS_FRACTION * 100:.0f}% IS / {(1 - WALK_FORWARD_IS_FRACTION) * 100:.0f}% OOS")

## Step 1: Load FX Sentiment Indices from Zip Files

Each country has a zip archive in `FX_SENTIMENT_DIR`. The zip is doubly partitioned by `topic_name` and `index_type`, with gzipped CSV part-files at the leaf level. These are **pre-aggregated at the hourly level** — filtering has already been applied upstream.

**Zip naming:** `macro_geopolitical_sentiment_{uuid}_{Country Name}.zip` — matched by the country name suffix (case-insensitive, spaces preserved). The UUID prefix is ignored during lookup. Where multiple zips exist for the same country (different UUIDs), the most recently modified file is used.

**Derived columns computed on load:**
- `topic_name` and `index_type` — extracted from the Hive partition path
- `sentiment_avg` — computed as `sentiment_sum / headline_count` (0 where `headline_count == 0`)
- `country` — inferred from the matched zip filename

In [ ]:
def find_zip_for_country(country: str) -> Path | None:
    """Find the most recent zip file for a country (by modification time)."""
    country_norm = country.lower().strip()
    candidates = [
        f
        for f in FX_SENTIMENT_DIR.iterdir()
        if f.suffix == ".zip" and (f.stem.lower().endswith(f"_{country_norm}") or f.stem.lower() == country_norm)
    ]
    if not candidates:
        return None
    return max(candidates, key=lambda f: f.stat().st_mtime)


def load_fx_sentiment(zip_path: Path, country: str) -> pd.DataFrame:
    """
    Load all topic/index_type partitions from a country zip archive.

    Zip structure:
        topic_name={Category}-{SubTopic}/index_type={DOMESTIC|INTERNATIONAL}/part-XXXXX-{uuid}.c000.csv.gz

    Raw CSV columns: publication_time, headline_count, sentiment_sum
    Derived on load: topic_name, index_type, sentiment_avg, country
    """
    frames = []
    with zipfile.ZipFile(zip_path, "r") as zf:
        for name in zf.namelist():
            if not name.endswith(".gz"):
                continue
            parts = name.split("/")
            if len(parts) < 3:
                continue
            topic_part, index_part = parts[0], parts[1]
            if not topic_part.startswith("topic_name=") or not index_part.startswith("index_type="):
                continue
            topic_name = topic_part.replace("topic_name=", "")
            index_type = index_part.replace("index_type=", "")
            with zf.open(name) as zipped_file:
                with gzip.open(zipped_file) as gz:
                    df = pd.read_csv(gz)
            df["topic_name"] = topic_name
            df["index_type"] = index_type
            frames.append(df)

    if not frames:
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True)
    combined["publication_time"] = pd.to_datetime(combined["publication_time"], format="ISO8601", utc=True)
    combined["sentiment_avg"] = np.where(
        combined["headline_count"] > 0,
        combined["sentiment_sum"] / combined["headline_count"],
        0.0,
    )
    combined["country"] = country
    return combined


# ── Load all countries ─────────────────────────────────────────────────────────
all_raw_sentiment = {}  # {country: DataFrame}

if not FX_SENTIMENT_DIR.exists():
    print(f"WARNING: {FX_SENTIMENT_DIR} does not exist — place country zip archives there.")
else:
    for country in COUNTRIES:
        zp = find_zip_for_country(country)
        if zp is None:
            print(f"WARNING: No zip found for '{country}' in {FX_SENTIMENT_DIR}")
            continue
        df = load_fx_sentiment(zp, country)
        if df.empty:
            print(f"WARNING: Empty data for '{country}' in {zp.name}")
            continue
        all_raw_sentiment[country] = df
        print(
            f"{COUNTRY_LABELS.get(country, country):20s}: {len(df):>9,} hourly rows | "
            f"topics: {df['topic_name'].nunique():2d} | index_types: {sorted(df['index_type'].unique())} | "
            f"date range: {df['publication_time'].min().date()} to {df['publication_time'].max().date()}"
        )

sample_country = list(all_raw_sentiment.keys())[0] if all_raw_sentiment else None
if sample_country:
    print(f"\nSample hourly data ({sample_country}):")
    ipy_display(
        all_raw_sentiment[sample_country][
            ["publication_time", "topic_name", "index_type", "sentiment_avg", "headline_count"]
        ].head(8)
    )

## Step 2: Load & Prepare FX Prices

We load the FX price file, match each country to its primary FX pair via `COUNTRY_TO_FX_PAIR`, compute **forward returns** at multiple horizons (7d, 1m, 3m), and classify each day into an **UP or DOWN regime** based on the trailing 63-day return sign.

In [ ]:
def load_and_prepare_fx_prices(country: str, fx_prices: pd.DataFrame) -> pd.DataFrame | None:
    """
    Load hourly FX OHLC data for a country's ticker, aggregate to daily close,
    then compute forward returns and UP/DOWN regime labels.

    fx_prices.csv columns: ticker, date (hourly), ask_open/high/low/close,
                            bid_open/high/low/close, volume, expiry
    Price used: ask_close (daily last bar of each session).
    """
    ticker = COUNTRY_TO_FX_PAIR.get(country)
    if ticker is None:
        return None
    subset = fx_prices[fx_prices["ticker"] == ticker].copy()
    if subset.empty:
        return None

    subset["trade_date"] = pd.to_datetime(subset["date"]).dt.normalize()
    # Take the last hourly bar of each trading day as the daily close
    daily = (
        subset.sort_values("date")
        .groupby("trade_date")
        .last()
        .reset_index()[["trade_date", "ask_close"]]
        .rename(columns={"ask_close": "price"})
        .sort_values("trade_date")
        .set_index("trade_date")
    )

    for label, days in RETURN_HORIZONS.items():
        daily[f"fwd_ret_{label}"] = daily["price"].shift(-days) / daily["price"] - 1

    trailing = daily["price"] / daily["price"].shift(REGIME_LOOKBACK) - 1
    daily["regime"] = np.where(trailing >= 0, "UP", "DOWN")
    daily.loc[trailing.isna(), "regime"] = np.nan
    return daily


raw_fx_prices = pd.read_csv(FX_PRICES_FILE, parse_dates=["date"])
all_fx_prices = {}

for country in COUNTRIES:
    pt = load_and_prepare_fx_prices(country, raw_fx_prices)
    if pt is None:
        print(f"WARNING: No FX price data for '{country}' (ticker: {COUNTRY_TO_FX_PAIR.get(country)})")
        continue
    all_fx_prices[country] = pt
    rc = pt["regime"].value_counts()
    ticker = COUNTRY_TO_FX_PAIR[country]
    print(
        f"{COUNTRY_LABELS.get(country, country):20s} [{ticker}]: "
        f"{len(pt)} price days | "
        f"UP={rc.get('UP', 0)} DOWN={rc.get('DOWN', 0)} | "
        f"range: {pt.index.min().date()} to {pt.index.max().date()}"
    )

if sample_country and sample_country in all_fx_prices:
    print(f"\nSample prices ({sample_country}):")
    ipy_display(all_fx_prices[sample_country][["price", "fwd_ret_7d", "fwd_ret_1m", "regime"]].head(5))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

countries_with_prices = [c for c in COUNTRIES if c in all_fx_prices]
n = len(countries_with_prices)
ncols = 2
nrows = (n + 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows), sharex=False)
axes = axes.flatten()

for ax, country in zip(axes, countries_with_prices):
    pt = all_fx_prices[country]
    ticker = COUNTRY_TO_FX_PAIR[country]
    label = COUNTRY_LABELS.get(country, country)

    ax.plot(pt.index, pt["price"], linewidth=0.9, color="#2563eb")
    ax.set_title(f"{label}  [{ticker}]", fontsize=10, fontweight="bold")
    ax.set_ylabel("Price", fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.tick_params(axis="x", labelsize=8, rotation=30)
    ax.tick_params(axis="y", labelsize=8)
    ax.grid(True, linestyle="--", alpha=0.4)

for ax in axes[n:]:
    ax.set_visible(False)

fig.suptitle("FX Currency Index Daily Prices", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


## Step 3: Aggregate Hourly Indices to Daily Sentiment

The raw sentiment data is at hourly granularity. We aggregate to daily, deriving **3 daily metrics** and splitting into two topic levels:

- **GROUP**: the prefix before the first `-` (e.g. `Economic Data`, `Politics`)
- **TOPIC**: the full topic name (e.g. `Economic Data-Inflation`, `Politics-Election`)

| Metric | Formula | Measures |
|---|---|---|
| `sentiment_avg` | `sentiment_sum / headline_count` per day | Average direction |
| `sentiment_std_daily` | std of hourly `sentiment_avg` values | Intra-day disagreement |
| `headline_count` | sum of hourly counts | Attention / volume |

In [ ]:
def aggregate_to_daily(raw: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate hourly FX sentiment indices to daily, deriving all metrics and
    splitting into GROUP and TOPIC levels.
    """
    raw = raw.copy()
    # Strip timezone so date is naive and merges cleanly with FX price dates
    raw["date"] = raw["publication_time"].dt.normalize().dt.tz_convert(None)

    daily = (
        raw.groupby(["date", "index_type", "topic_name"])
        .agg(
            headline_count=("headline_count", "sum"),
            sentiment_sum=("sentiment_sum", "sum"),
            sentiment_std_daily=("sentiment_avg", "std"),
        )
        .reset_index()
    )
    daily["sentiment_avg"] = np.where(
        daily["headline_count"] > 0,
        daily["sentiment_sum"] / daily["headline_count"],
        0.0,
    )
    daily["sentiment_std_daily"] = daily["sentiment_std_daily"].fillna(0)
    daily["group"] = daily["topic_name"].str.split("-").str[0].str.strip()

    # Group-level aggregation
    group_agg = (
        daily.groupby(["date", "index_type", "group"])
        .agg(
            headline_count=("headline_count", "sum"),
            sentiment_sum=("sentiment_sum", "sum"),
            sentiment_std_daily=("sentiment_std_daily", "mean"),
        )
        .reset_index()
    )
    group_agg["sentiment_avg"] = np.where(
        group_agg["headline_count"] > 0,
        group_agg["sentiment_sum"] / group_agg["headline_count"],
        0.0,
    )
    group_agg.rename(columns={"group": "name"}, inplace=True)
    group_agg["topic_level"] = "GROUP"

    topic_part = daily.rename(columns={"topic_name": "name"}).copy()
    topic_part["topic_level"] = "TOPIC"

    common_cols = ["date", "index_type", "name", "topic_level"] + SENTIMENT_METRICS
    return pd.concat(
        [
            group_agg[[c for c in common_cols if c in group_agg.columns]],
            topic_part[[c for c in common_cols if c in topic_part.columns]],
        ],
        ignore_index=True,
    )


all_sentiment = {}  # {country: DataFrame}
for country, df in all_raw_sentiment.items():
    sent = aggregate_to_daily(df)
    all_sentiment[country] = sent
    n_idx = sent.groupby(["topic_level", "index_type", "name"]).ngroups
    print(f"{COUNTRY_LABELS.get(country, country):20s}: {n_idx} unique sentiment indices, {len(sent):,} daily rows")

if sample_country:
    print(f"\nSample daily sentiment ({sample_country}):")
    display_cols = ["date", "index_type", "name", "topic_level"] + SENTIMENT_METRICS
    ipy_display(all_sentiment[sample_country][display_cols].head(8))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Plot one panel per country: daily sentiment_avg for BOTH index types,
# averaged across all topics at GROUP level, so the chart stays readable.
countries_with_sent = [c for c in COUNTRIES if c in all_sentiment]
n = len(countries_with_sent)
ncols = 2
nrows = (n + 1) // ncols

_COLORS = {"DOMESTIC": "#2563eb", "INTERNATIONAL": "#dc2626"}

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.5 * nrows), sharex=False)
axes = axes.flatten()

for ax, country in zip(axes, countries_with_sent):
    sent = all_sentiment[country]
    label = COUNTRY_LABELS.get(country, country)

    # Average sentiment_avg across all GROUP-level topics per day and index_type
    grp = sent[sent["topic_level"] == "GROUP"].groupby(["date", "index_type"])["sentiment_avg"].mean().reset_index()

    for itype, color in _COLORS.items():
        sub = grp[grp["index_type"] == itype].sort_values("date")
        if sub.empty:
            continue
        # 21-day rolling mean to reduce daily noise
        smoothed = sub.set_index("date")["sentiment_avg"].rolling(21, min_periods=1).mean()
        ax.plot(smoothed.index, smoothed.values, linewidth=0.9, color=color, label=itype)

    ax.axhline(0, linewidth=0.7, color="black", linestyle="--", alpha=0.5)
    ax.set_title(f"{label}", fontsize=10, fontweight="bold")
    ax.set_ylabel("Sentiment avg\n(21d roll mean)", fontsize=7)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.tick_params(axis="x", labelsize=8, rotation=30)
    ax.tick_params(axis="y", labelsize=8)
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(fontsize=7, loc="upper left", framealpha=0.6)

for ax in axes[n:]:
    ax.set_visible(False)

fig.suptitle(
    "Daily Macro Sentiment Avg by Country & Index Type\n(GROUP-level topics, 21-day rolling mean)",
    fontsize=12,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
plt.show()


## Step 4: Coverage & Correlation Analysis

For every combination of `(regime, metric, topic_level, index_type, name, transform, window, return_horizon)` we:

1. Apply a **rolling transform** to the daily sentiment series
2. Align with forward FX price returns
3. Compute **Pearson correlation** and a two-tailed **t-test** p-value

### Transforms

- **Rolling z-score** — normalises sentiment relative to its own recent history (stationary, primary transform)
- **Rolling mean** — smooths daily noise (secondary check)

### Coverage

Not every topic has data every day. Indices with coverage below **15%** of business days are flagged as **sparse** but still included.

In [ ]:
def compute_coverage(sentiment_all: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (tl, it, nm), grp in sentiment_all.groupby(["topic_level", "index_type", "name"]):
        bdays = len(pd.bdate_range(grp["date"].min(), grp["date"].max()))
        n_days = grp["date"].nunique()
        rows.append(
            {
                "topic_level": tl,
                "index_type": it,
                "name": nm,
                "days_with_data": n_days,
                "total_bdays": bdays,
                "coverage_pct": round(100 * n_days / max(bdays, 1), 1),
            }
        )
    return pd.DataFrame(rows)


def apply_transforms(series: pd.Series, window: int) -> dict[str, pd.Series]:
    roll_mean = series.rolling(window, min_periods=max(1, window // 2)).mean()
    roll_std = series.rolling(window, min_periods=max(1, window // 2)).std()
    return {
        "rolling_zscore": (series - roll_mean) / roll_std.replace(0, np.nan),
        "rolling_mean": roll_mean,
    }


def vectorized_correlation_sweep(sentiment_all: pd.DataFrame, prices_country: pd.DataFrame) -> pd.DataFrame:
    fwd_ret_cols = [f"fwd_ret_{h}" for h in RETURN_HORIZONS]
    horizon_names = list(RETURN_HORIZONS.keys())
    prices_indexed = prices_country[["regime"] + fwd_ret_cols].copy()
    regime_col = prices_indexed["regime"]
    ret_matrix = prices_indexed[fwd_ret_cols].values
    price_index = prices_indexed.index

    results = []
    for (topic_level, index_type, name), grp in sentiment_all.groupby(["topic_level", "index_type", "name"]):
        grp = grp.set_index("date").sort_index()
        full_idx = pd.date_range(grp.index.min(), grp.index.max(), freq="B")

        for metric in SENTIMENT_METRICS:
            if metric not in grp.columns:
                continue
            raw_series = grp[metric].reindex(full_idx).fillna(0)
            if raw_series.dropna().nunique() < 3:
                continue

            for window in ROLLING_WINDOWS:
                transformed = apply_transforms(raw_series, window)
                for transform_name, t_series in transformed.items():
                    common_idx = t_series.dropna().index.intersection(price_index)
                    if len(common_idx) < 30:
                        continue
                    s_vals = t_series.loc[common_idx].values
                    p_loc = price_index.get_indexer(common_idx)
                    r_vals = ret_matrix[p_loc]
                    reg_vals = regime_col.values[p_loc]

                    for regime in REGIMES:
                        rmask = np.ones(len(common_idx), dtype=bool) if regime == "BOTH" else (reg_vals == regime)
                        s_reg, r_reg = s_vals[rmask], r_vals[rmask]
                        if rmask.sum() < 30:
                            continue
                        s_finite = np.isfinite(s_reg)
                        for h_idx, horizon in enumerate(horizon_names):
                            r_col = r_reg[:, h_idx]
                            valid = s_finite & np.isfinite(r_col)
                            n_h = valid.sum()
                            if n_h < 30:
                                continue
                            sx, ry = s_reg[valid], r_col[valid]
                            sx_m, ry_m = sx - sx.mean(), ry - ry.mean()
                            ss_x, ss_y = (sx_m**2).sum(), (ry_m**2).sum()
                            if ss_x == 0 or ss_y == 0:
                                continue
                            corr = np.clip((sx_m * ry_m).sum() / np.sqrt(ss_x * ss_y), -1, 1)
                            if abs(corr) < 1:
                                t_stat = corr * np.sqrt((n_h - 2) / (1 - corr**2))
                                pval = 2 * t_dist.sf(abs(t_stat), n_h - 2)
                            else:
                                t_stat, pval = np.inf, 0.0
                            results.append(
                                (
                                    regime,
                                    metric,
                                    topic_level,
                                    index_type,
                                    name,
                                    transform_name,
                                    window,
                                    horizon,
                                    int(n_h),
                                    round(corr, 6),
                                    pval,
                                    round(t_stat, 4),
                                )
                            )

    if not results:
        return pd.DataFrame()
    return pd.DataFrame(
        results,
        columns=[
            "regime",
            "metric",
            "topic_level",
            "index_type",
            "name",
            "transform",
            "window",
            "return_horizon",
            "n_obs",
            "pearson_r",
            "p_value",
            "t_stat",
        ],
    )


all_corr_results = {}
all_coverage = {}

for country in COUNTRIES:
    if country not in all_sentiment or country not in all_fx_prices:
        continue
    print(f"\n--- {COUNTRY_LABELS.get(country, country)} ---")
    t0 = time.perf_counter()
    cov = compute_coverage(all_sentiment[country])
    all_coverage[country] = cov
    res = vectorized_correlation_sweep(all_sentiment[country], all_fx_prices[country])
    res["country"] = country
    cov_map = cov.set_index(["topic_level", "index_type", "name"])["coverage_pct"]
    res["coverage_pct"] = res.set_index(["topic_level", "index_type", "name"]).index.map(cov_map).values
    res["coverage_pct"] = res["coverage_pct"].fillna(0)
    all_corr_results[country] = res
    elapsed = time.perf_counter() - t0
    n_sparse = (cov["coverage_pct"] < SPARSE_THRESHOLD_PCT).sum()
    print(
        f"  {len(res):,} correlation tests | {len(cov)} indices × {len(SENTIMENT_METRICS)} metrics "
        f"({n_sparse} sparse) | {elapsed:.1f}s"
    )

total = sum(len(v) for v in all_corr_results.values())
print(f"\nTotal tests across all countries: {total:,}")

## Step 5: Apply Significance Testing (Benjamini-Hochberg FDR)

### The multiple-testing problem

With thousands of correlation tests per country, many false positives are expected at p < 0.05. For example, 8,000 tests × 5% = 400 expected false positives under the null.

### How we address it

1. **Raw p-value** threshold at 0.05 (labelled `significant`)
2. **Benjamini-Hochberg (BH) FDR correction** — applied per regime to control the false discovery rate (labelled `significant (BH adjusted)`)

### How to read the output

- **BH sig**: number of tests surviving the BH correction
- **Expected FP**: 5% × total tests (what we'd expect by chance)
- **Excess**: BH sig − Expected FP (positive = more signal than noise)

In [ ]:
def benjamini_hochberg(p_values: np.ndarray, alpha: float = 0.05) -> np.ndarray:
    n = len(p_values)
    if n == 0:
        return np.array([], dtype=bool)
    sorted_idx = np.argsort(p_values)
    sorted_p = p_values[sorted_idx]
    thresholds = alpha * np.arange(1, n + 1) / n
    below = sorted_p <= thresholds
    if not below.any():
        return np.zeros(n, dtype=bool)
    max_idx = np.max(np.where(below))
    significant = np.zeros(n, dtype=bool)
    significant[sorted_idx[: max_idx + 1]] = True
    return significant


def apply_significance(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    significance = pd.Series("not significant", index=df.index)
    significance[df["p_value"] < SIGNIFICANCE_LEVEL] = "significant"
    for regime in REGIMES:
        regime_idx = df.index[df["regime"] == regime]
        if len(regime_idx) == 0:
            continue
        bh_mask = benjamini_hochberg(df.loc[regime_idx, "p_value"].values, SIGNIFICANCE_LEVEL)
        significance.loc[regime_idx[bh_mask]] = "significant (BH adjusted)"
    df["significance"] = significance
    return df


for country in COUNTRIES:
    if country not in all_corr_results:
        continue
    all_corr_results[country] = apply_significance(all_corr_results[country])
    res = all_corr_results[country]
    total_tests = len(res)
    n_bh = (res["significance"] == "significant (BH adjusted)").sum()
    expected_fp = total_tests * SIGNIFICANCE_LEVEL
    print(
        f"\n{COUNTRY_LABELS.get(country, country)}: {total_tests:,} tests | "
        f"BH sig: {n_bh:,} ({100 * n_bh / max(total_tests, 1):.1f}%) | "
        f"Expected FP: {expected_fp:.0f} | Excess: {n_bh - expected_fp:+.0f}"
    )
    bh = res[res["significance"] == "significant (BH adjusted)"]
    for metric in SENTIMENT_METRICS:
        m = bh[bh["metric"] == metric]
        m_total = (res["metric"] == metric).sum()
        max_r = m["pearson_r"].abs().max() if not m.empty else 0
        print(
            f"  {metric:<28s}: {len(m):>5,} BH-sig / {m_total:>5,} "
            f"({100 * len(m) / max(m_total, 1):.1f}%)  max|r|={max_r:.4f}"
        )

## Step 6: Index Summary & Walk-Forward Validation

For each `(topic_level, index_type, name)` we build a summary row with:

- Coverage stats and a sparse flag
- Per-regime best correlation, p-value, significant horizons, and **which metric** produced it
- A **composite score** = Σ −log10(p) × |r| across regimes
- The **best_metric** — which metric produces the strongest single driver for this index

Then we run a **60/40 walk-forward split**: find significant drivers in the in-sample period, and check how many retain BH-significance out-of-sample. The walk-forward key includes the `metric` dimension so each metric is validated independently.

In [ ]:
def build_country_summary(results_df: pd.DataFrame, coverage_df: pd.DataFrame, country: str) -> pd.DataFrame:
    sig = results_df[results_df["significance"] == "significant (BH adjusted)"].copy()
    if not sig.empty:
        sig["abs_r"] = sig["pearson_r"].abs()

    sig_grouped = {}
    if not sig.empty:
        for (tl, it, nm, regime), grp in sig.groupby(["topic_level", "index_type", "name", "regime"]):
            sig_grouped[(tl, it, nm, regime)] = grp

    horizon_order = list(RETURN_HORIZONS.keys())
    rows = []
    for _, cov_row in coverage_df.iterrows():
        tl, it, nm = cov_row["topic_level"], cov_row["index_type"], cov_row["name"]
        row = {
            "country": country,
            "topic_level": tl,
            "name": nm,
            "index_type": it,
            "days_with_data": cov_row["days_with_data"],
            "total_bdays": cov_row["total_bdays"],
            "coverage_pct": cov_row["coverage_pct"],
            "sparse": cov_row["coverage_pct"] < SPARSE_THRESHOLD_PCT,
        }
        score = 0.0
        for regime in REGIMES:
            r_sig = sig_grouped.get((tl, it, nm, regime))
            if r_sig is None or r_sig.empty:
                row[f"{regime}_sig_horizons"] = ""
                row[f"{regime}_best_r"] = np.nan
                row[f"{regime}_best_t"] = np.nan
                row[f"{regime}_best_p"] = np.nan
                row[f"{regime}_best_metric"] = ""
            else:
                horizons = sorted(r_sig["return_horizon"].unique(), key=lambda h: horizon_order.index(h))
                row[f"{regime}_sig_horizons"] = "|".join(horizons)
                best_idx = r_sig["abs_r"].idxmax()
                best_r = r_sig.loc[best_idx, "pearson_r"]
                best_p = r_sig.loc[best_idx, "p_value"]
                row[f"{regime}_best_r"] = round(best_r, 4)
                row[f"{regime}_best_t"] = r_sig.loc[best_idx, "t_stat"]
                row[f"{regime}_best_p"] = best_p
                row[f"{regime}_best_metric"] = r_sig.loc[best_idx, "metric"]
                score += -np.log10(max(best_p, 1e-300)) * abs(best_r)

        sig_regimes = [r for r in REGIMES if row.get(f"{r}_sig_horizons", "") != ""]
        row["sig_regimes"] = "|".join(sig_regimes) if sig_regimes else ""

        idx_sig = (
            sig[(sig["topic_level"] == tl) & (sig["index_type"] == it) & (sig["name"] == nm)]
            if not sig.empty
            else pd.DataFrame()
        )
        best_metric, best_abs_r = "", 0
        for metric in SENTIMENT_METRICS:
            m_sig = idx_sig[idx_sig["metric"] == metric] if not idx_sig.empty else pd.DataFrame()
            row[f"{metric}_n_sig"] = len(m_sig)
            if not m_sig.empty:
                m_best_idx = m_sig["abs_r"].idxmax()
                row[f"{metric}_best_r"] = round(m_sig.loc[m_best_idx, "pearson_r"], 4)
                if m_sig.loc[m_best_idx, "abs_r"] > best_abs_r:
                    best_abs_r = m_sig.loc[m_best_idx, "abs_r"]
                    best_metric = metric
            else:
                row[f"{metric}_best_r"] = np.nan
        row["best_metric"] = best_metric
        row["score"] = round(score, 2)
        rows.append(row)

    summary = pd.DataFrame(rows)
    summary.sort_values("score", ascending=False, inplace=True)
    summary.insert(0, "rank", range(1, len(summary) + 1))
    return summary


def walk_forward_validate(sentiment_all: pd.DataFrame, prices_country: pd.DataFrame, country: str) -> pd.DataFrame:
    all_dates = prices_country.index.sort_values()
    n_total = len(all_dates)
    n_is = int(n_total * WALK_FORWARD_IS_FRACTION)
    split_date = all_dates[n_is - 1]
    is_start, is_end = all_dates[0], split_date
    oos_start, oos_end = all_dates[n_is], all_dates[-1]

    is_sent = sentiment_all[(sentiment_all["date"] >= is_start) & (sentiment_all["date"] <= is_end)]
    oos_sent = sentiment_all[(sentiment_all["date"] >= oos_start) & (sentiment_all["date"] <= oos_end)]
    is_prices = prices_country.loc[is_start:is_end]
    oos_prices = prices_country.loc[oos_start:oos_end]

    is_results = apply_significance(vectorized_correlation_sweep(is_sent, is_prices))
    oos_results = apply_significance(vectorized_correlation_sweep(oos_sent, oos_prices))
    if is_results.empty or oos_results.empty:
        return pd.DataFrame()

    def _key(row):
        return (
            row["regime"],
            row["metric"],
            row["topic_level"],
            row["index_type"],
            row["name"],
            row["transform"],
            row["window"],
            row["return_horizon"],
        )

    is_sig = is_results[is_results["significance"] == "significant (BH adjusted)"].copy()
    is_sig["key"] = is_sig.apply(_key, axis=1)
    oos_results["key"] = oos_results.apply(_key, axis=1)
    oos_lookup = {row["key"]: row for _, row in oos_results.iterrows()}

    matched = []
    for _, is_row in is_sig.iterrows():
        k = is_row["key"]
        if k in oos_lookup:
            oos_row = oos_lookup[k]
            matched.append(
                {
                    "regime": is_row["regime"],
                    "metric": is_row["metric"],
                    "topic_level": is_row["topic_level"],
                    "index_type": is_row["index_type"],
                    "name": is_row["name"],
                    "transform": is_row["transform"],
                    "window": is_row["window"],
                    "return_horizon": is_row["return_horizon"],
                    "is_r": is_row["pearson_r"],
                    "is_p": is_row["p_value"],
                    "oos_r": oos_row["pearson_r"],
                    "oos_p": oos_row["p_value"],
                    "oos_sig": oos_row["significance"],
                }
            )
    return pd.DataFrame(matched)


def enrich_with_walkforward(idx_summary: pd.DataFrame, wf_df: pd.DataFrame) -> pd.DataFrame:
    idx_summary = idx_summary.copy()
    for col, val in {
        "wf_is_sig": 0,
        "wf_oos_persistent": 0,
        "wf_oos_pct": 0.0,
        "wf_same_sign": 0,
        "wf_sign_rate": 0.0,
        "wf_best_oos_r": np.nan,
        "wf_best_oos_p": np.nan,
    }.items():
        idx_summary[col] = val
    if wf_df.empty:
        return idx_summary
    for (tl, it, nm), grp in wf_df.groupby(["topic_level", "index_type", "name"]):
        mask = (idx_summary["topic_level"] == tl) & (idx_summary["index_type"] == it) & (idx_summary["name"] == nm)
        if not mask.any():
            continue
        n_is = len(grp)
        persistent = grp[grp["oos_sig"] == "significant (BH adjusted)"]
        n_oos = len(persistent)
        same_sign = (np.sign(grp["is_r"]) == np.sign(grp["oos_r"])).sum()
        idx_summary.loc[mask, "wf_is_sig"] = n_is
        idx_summary.loc[mask, "wf_oos_persistent"] = n_oos
        idx_summary.loc[mask, "wf_oos_pct"] = round(100 * n_oos / max(n_is, 1), 1)
        idx_summary.loc[mask, "wf_same_sign"] = same_sign
        idx_summary.loc[mask, "wf_sign_rate"] = round(100 * same_sign / max(n_is, 1), 1)
        if not persistent.empty:
            best_idx = persistent["oos_r"].abs().idxmax()
            idx_summary.loc[mask, "wf_best_oos_r"] = round(persistent.loc[best_idx, "oos_r"], 4)
            idx_summary.loc[mask, "wf_best_oos_p"] = persistent.loc[best_idx, "oos_p"]
    return idx_summary


all_idx_summaries = {}
for country in COUNTRIES:
    if country not in all_corr_results:
        continue
    print(f"\n--- {COUNTRY_LABELS.get(country, country)} ---")
    t0 = time.perf_counter()
    idx_sum = build_country_summary(all_corr_results[country], all_coverage[country], country)
    n_sig = (idx_sum["score"] > 0).sum()
    print(f"  Index summary: {len(idx_sum)} indices, {n_sig} significant")

    wf_df = walk_forward_validate(all_sentiment[country], all_fx_prices[country], country)
    n_is = len(wf_df)
    n_oos = (wf_df["oos_sig"] == "significant (BH adjusted)").sum() if not wf_df.empty else 0
    elapsed = time.perf_counter() - t0
    print(f"  WF: {n_is} IS-sig matched, {n_oos} persistent OOS ({100 * n_oos / max(n_is, 1):.1f}%) [{elapsed:.1f}s]")

    idx_sum = enrich_with_walkforward(idx_sum, wf_df)
    all_idx_summaries[country] = idx_sum

if sample_country and sample_country in all_idx_summaries:
    print(f"\nTop 5 drivers for {sample_country}:")
    ipy_display(
        all_idx_summaries[sample_country][
            ["rank", "name", "index_type", "score", "best_metric", "BOTH_best_r", "wf_oos_persistent"]
        ].head(5)
    )

## Step 7: Cross-Country Assessment

We now aggregate results across all countries into three summary tables:

1. **Overall Ranking** — composite grade (A–F) based on BH significance, correlation strength, OOS persistence, and dominant metric per country
2. **Top FX Drivers Per Country** — the single highest-scoring topic for each country, showing which metric produced it
3. **Top Sentiment Group Drivers Per Country** — group-level aggregation showing breadth of signal and best metric per group

In [ ]:
def load_fx_summary_from_country(country: str, idx: pd.DataFrame, results_df: pd.DataFrame) -> dict:
    n_indices = len(idx)
    n_sig = (idx["score"] > 0).sum()

    best_r_both = idx["BOTH_best_r"].abs().max() if "BOTH_best_r" in idx else np.nan
    best_r_up = idx["UP_best_r"].abs().max() if "UP_best_r" in idx else np.nan
    best_r_down = idx["DOWN_best_r"].abs().max() if "DOWN_best_r" in idx else np.nan
    best_r_any = np.nanmax([best_r_both, best_r_up, best_r_down])

    top_score = idx["score"].max()
    top_row = idx.loc[idx["score"].idxmax()] if top_score > 0 else None
    top_driver = top_row["name"] if top_row is not None else "--"
    top_idx_type = top_row["index_type"] if top_row is not None else "--"
    top_best_metric = top_row["best_metric"] if top_row is not None and "best_metric" in idx else "--"

    sig_idx = idx[idx["score"] > 0]
    metric_wins = sig_idx["best_metric"].value_counts() if "best_metric" in sig_idx else pd.Series(dtype=int)
    dominant_metric = metric_wins.index[0] if len(metric_wins) > 0 else "--"

    total_tests = len(results_df)
    n_bh = (results_df["significance"] == "significant (BH adjusted)").sum()

    wf_is = int(idx["wf_is_sig"].sum()) if "wf_is_sig" in idx else 0
    wf_oos = int(idx["wf_oos_persistent"].sum()) if "wf_oos_persistent" in idx else 0
    wf_pct = round(100 * wf_oos / max(wf_is, 1), 1) if wf_is > 0 else 0.0
    wf_sign = int(idx["wf_same_sign"].sum()) if "wf_same_sign" in idx else 0
    wf_sign_pct = round(100 * wf_sign / max(wf_is, 1), 1) if wf_is > 0 else 0.0

    best_oos_r = np.nan
    if "wf_best_oos_r" in idx:
        valid = idx["wf_best_oos_r"].dropna()
        if not valid.empty:
            best_oos_r = valid.abs().max()

    sig_regimes = idx[idx["score"] > 0]["sig_regimes"].value_counts()
    n_all_regime = sum(1 for v in sig_regimes.index if "BOTH" in v and "UP" in v and "DOWN" in v)
    n_regime_specific = n_sig - n_all_regime

    return {
        "country": country,
        "label": COUNTRY_LABELS.get(country, country),
        "fx_pair": COUNTRY_TO_FX_PAIR.get(country, "--"),
        "n_indices": n_indices,
        "total_corr_tests": total_tests,
        "n_bh_sig": n_bh,
        "bh_sig_pct": round(100 * n_bh / max(total_tests, 1), 1),
        "n_sig_indices": n_sig,
        "top_score": top_score,
        "top_driver": top_driver,
        "top_idx_type": top_idx_type,
        "top_best_metric": top_best_metric,
        "dominant_metric": dominant_metric,
        "best_r_any": round(best_r_any, 4) if not np.isnan(best_r_any) else np.nan,
        "best_r_both": round(best_r_both, 4) if not np.isnan(best_r_both) else np.nan,
        "wf_is_sig": wf_is,
        "wf_oos_persistent": wf_oos,
        "wf_oos_pct": wf_pct,
        "wf_sign_consistency": wf_sign_pct,
        "best_oos_r": round(best_oos_r, 4) if not np.isnan(best_oos_r) else np.nan,
        "n_all_regime_sig": n_all_regime,
        "n_regime_specific": n_regime_specific,
    }


def compute_overall_grade(row: dict) -> tuple[str, float]:
    score = 0.0
    score += min(row.get("bh_sig_pct", 0) / 30.0, 1.0) * 10
    score += min(row.get("n_sig_indices", 0) / 80.0, 1.0) * 10
    score += min(row.get("top_score", 0) / 50.0, 1.0) * 15
    best_r = row.get("best_r_any", 0) or 0
    score += min(best_r / 0.5, 1.0) * 10
    score += min(row.get("wf_oos_pct", 0) / 50.0, 1.0) * 30
    score += min(row.get("wf_sign_consistency", 0) / 80.0, 1.0) * 10
    oos_r = row.get("best_oos_r", 0) or 0
    score += min(oos_r / 0.5, 1.0) * 15
    grade = "A" if score >= 80 else "B" if score >= 65 else "C" if score >= 45 else "D" if score >= 25 else "F"
    return grade, round(score, 1)


def load_group_drivers(country: str, idx: pd.DataFrame) -> pd.DataFrame:
    sig = idx[idx["score"] > 0].copy()
    if sig.empty:
        return pd.DataFrame()
    sig["group"] = sig["name"].str.split("-").str[0].str.strip()
    rows = []
    for group, grp in sig.groupby("group"):
        best_r_cols = [c for c in ["BOTH_best_r", "UP_best_r", "DOWN_best_r"] if c in grp]
        best_r = grp[best_r_cols].abs().max().max() if best_r_cols else np.nan
        best_score_idx = grp["score"].idxmax()
        best_oos_r = np.nan
        if "wf_best_oos_r" in grp:
            valid = grp["wf_best_oos_r"].dropna()
            if not valid.empty:
                best_oos_r = valid.abs().max()
        wf_oos = int(grp["wf_oos_persistent"].sum()) if "wf_oos_persistent" in grp else 0
        wf_is = int(grp["wf_is_sig"].sum()) if "wf_is_sig" in grp else 0
        rows.append(
            {
                "country": country,
                "label": COUNTRY_LABELS.get(country, country),
                "group": group,
                "n_sig": len(grp),
                "top_score": grp["score"].max(),
                "best_r": round(best_r, 4) if not np.isnan(best_r) else np.nan,
                "best_driver": grp.loc[best_score_idx, "name"],
                "best_idx_type": grp.loc[best_score_idx, "index_type"],
                "best_metric": grp.loc[best_score_idx, "best_metric"] if "best_metric" in grp else "--",
                "wf_is_sig": wf_is,
                "wf_oos_persistent": wf_oos,
                "wf_oos_pct": round(100 * wf_oos / max(wf_is, 1), 1) if wf_is > 0 else 0.0,
                "best_oos_r": round(best_oos_r, 4) if not np.isnan(best_oos_r) else np.nan,
            }
        )
    return pd.DataFrame(rows).sort_values("top_score", ascending=False)


ct_rows = []
for country in COUNTRIES:
    if country not in all_idx_summaries:
        continue
    row = load_fx_summary_from_country(country, all_idx_summaries[country], all_corr_results[country])
    grade, score = compute_overall_grade(row)
    row["grade"] = grade
    row["overall_score"] = score
    ct_rows.append(row)

ct_df = pd.DataFrame(ct_rows)
ct_df.sort_values("overall_score", ascending=False, inplace=True)
ct_df.reset_index(drop=True, inplace=True)
ct_df.insert(0, "rank", range(1, len(ct_df) + 1))

group_frames = []
for country in COUNTRIES:
    if country not in all_idx_summaries:
        continue
    gdf = load_group_drivers(country, all_idx_summaries[country])
    if not gdf.empty:
        group_frames.append(gdf)
all_groups = pd.concat(group_frames, ignore_index=True) if group_frames else pd.DataFrame()

print("Cross-country assessment built successfully.")
print(f"Countries: {len(ct_df)}, Group driver rows: {len(all_groups)}")

### Table 1: Overall Ranking

Each country receives a composite grade (A–F) based on:

- **BH %**: Percentage of tests that are BH-significant (signal density)
- **Sig Idx**: Number of sentiment indices with at least one significant result
- **Best |r|**: Strongest absolute Pearson correlation across all regimes
- **OOS %**: Percentage of IS-significant drivers that persist out-of-sample
- **Sign %**: Percentage of matched drivers with consistent sign IS vs OOS
- **OOS |r|**: Strongest OOS correlation among persistent drivers

In [ ]:
display_cols_1 = [
    "rank",
    "country",
    "label",
    "fx_pair",
    "grade",
    "overall_score",
    "n_sig_indices",
    "bh_sig_pct",
    "best_r_any",
    "wf_oos_pct",
    "wf_sign_consistency",
    "best_oos_r",
    "dominant_metric",
]
ct_df[display_cols_1].rename(
    columns={
        "rank": "#",
        "label": "Country",
        "fx_pair": "FX Pair",
        "grade": "Grade",
        "overall_score": "Score",
        "n_sig_indices": "Sig Idx",
        "bh_sig_pct": "BH %",
        "best_r_any": "Best |r|",
        "wf_oos_pct": "OOS %",
        "wf_sign_consistency": "Sign %",
        "best_oos_r": "OOS |r|",
    }
)

### Table 2: Top FX Drivers Per Country

The single highest-scoring sentiment topic for each country, with its composite score and Pearson correlation strength.

In [ ]:
display_cols_2 = [
    "rank",
    "country",
    "top_score",
    "top_driver",
    "top_idx_type",
    "top_best_metric",
    "best_r_both",
]
ct_df[display_cols_2].rename(
    columns={
        "rank": "#",
        "top_score": "Top Score",
        "top_driver": "Top FX Driver",
        "top_idx_type": "Index Type",
        "top_best_metric": "Best Metric",
        "best_r_both": "|r| All",
    }
)

### Table 3: Top Sentiment Group Drivers Per Country

Group-level aggregation (prefix before the first `-`, e.g. `Economic Data`, `Politics`) showing the breadth and depth of FX signal within each sentiment category.

In [ ]:
if not all_groups.empty:
    display_cols_3 = [
        "country",
        "label",
        "group",
        "n_sig",
        "top_score",
        "best_r",
        "best_metric",
        "best_driver",
        "wf_oos_persistent",
        "wf_oos_pct",
        "best_oos_r",
    ]
    ipy_display(
        all_groups[display_cols_3].rename(
            columns={
                "label": "Country",
                "n_sig": "Sig",
                "top_score": "Score",
                "best_r": "Best |r|",
                "best_driver": "Best FX Driver",
                "wf_oos_persistent": "OOS Sig",
                "wf_oos_pct": "OOS %",
                "best_oos_r": "OOS |r|",
            }
        )
    )
else:
    print("No group drivers found.")

### Regime Insights

Some sentiment drivers are significant across all market conditions, while others only work in **UP** (bullish) or **DOWN** (bearish) regimes.

- **Regime-universal drivers** are the most robust — they work regardless of the market environment
- **Regime-specific drivers** (UP-only or DOWN-only) still carry signal but should be deployed conditionally
- **Sign-flipping drivers** — where the correlation reverses between UP and DOWN regimes — are particularly interesting for regime-conditional strategies

In [ ]:
regime_rows = []
for country in COUNTRIES:
    if country not in all_idx_summaries:
        continue
    idx = all_idx_summaries[country]
    sig = idx[idx["score"] > 0].copy()
    if sig.empty:
        continue

    up_only = sig[sig["sig_regimes"].str.contains("UP", na=False) & ~sig["sig_regimes"].str.contains("DOWN", na=False)]
    down_only = sig[
        sig["sig_regimes"].str.contains("DOWN", na=False) & ~sig["sig_regimes"].str.contains("UP", na=False)
    ]
    both_ud = sig[sig["sig_regimes"].str.contains("UP", na=False) & sig["sig_regimes"].str.contains("DOWN", na=False)]

    n_sign_flip = 0
    flip_examples = []
    for _, row in both_ud.iterrows():
        up_r, down_r = row.get("UP_best_r"), row.get("DOWN_best_r")
        if pd.notna(up_r) and pd.notna(down_r) and up_r * down_r < 0:
            n_sign_flip += 1
            if len(flip_examples) < 2:
                flip_examples.append(f"{row['name']} (UP r={up_r:+.3f}, DOWN r={down_r:+.3f})")

    regime_rows.append(
        {
            "country": country,
            "label": COUNTRY_LABELS.get(country, country),
            "fx_pair": COUNTRY_TO_FX_PAIR.get(country, "--"),
            "total_sig": len(sig),
            "UP only": len(up_only),
            "DOWN only": len(down_only),
            "both UP & DOWN": len(both_ud),
            "sign flips": n_sign_flip,
            "flip_examples": "; ".join(flip_examples) if flip_examples else "--",
        }
    )

regime_df = pd.DataFrame(regime_rows)
print("Regime Breakdown: How many significant FX drivers are regime-specific vs universal?\n")
ipy_display(
    regime_df[
        [
            "country",
            "label",
            "fx_pair",
            "total_sig",
            "UP only",
            "DOWN only",
            "both UP & DOWN",
            "sign flips",
        ]
    ].rename(columns={"label": "Country", "total_sig": "Total Sig"})
)

flips = regime_df[regime_df["sign flips"] > 0]
if not flips.empty:
    print("\nSign-Flipping Drivers (correlation reverses between UP and DOWN regimes):\n")
    for _, row in flips.iterrows():
        print(f"  {row['country']} ({row['label']}): {row['sign flips']} driver(s)")
        if row["flip_examples"] != "--":
            for ex in row["flip_examples"].split("; "):
                print(f"    \u2192 {ex}")
else:
    print("\nNo sign-flipping FX drivers detected across any country.")